# LightGBM Hyperparameter Tuning with Optuna

This notebook focus on optimizing the LightGBM classifier using Bayesian search for multiclass churn prediction.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import joblib
import os
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

d:\Proj\Churn analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_data():
    logger.info("Loading split data...")
    X_train = pd.read_csv('../../data/04_features/X_train.csv')
    y_train = pd.read_csv('../../data/04_features/y_train.csv').squeeze().astype(int)
    X_test = pd.read_csv('../../data/04_features/X_test.csv')
    y_test = pd.read_csv('../../data/04_features/y_test.csv').squeeze().astype(int)
    return X_train, X_test, y_train, y_test

X_train_full, X_test, y_train_full, y_test = load_data()

# Create a validation set for Optuna and Early Stopping
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

logger.info(f"Train set: {X_train.shape}, Val set: {X_val.shape}, Test set: {X_test.shape}")

2026-04-15 00:41:30,698 - INFO - Loading split data...
2026-04-15 00:41:30,907 - INFO - Train set: (4096, 24), Val set: (1025, 24), Test set: (1281, 24)


In [3]:
def objective(trial):
    param = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'class_weight': 'balanced',
        'n_estimators': trial.suggest_int('n_estimators', 200, 800),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 10, 50),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
    }

    model = lgb.LGBMClassifier(**param)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='multi_logloss',
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False)
        ]
    )

    preds = model.predict(X_val)
    score = f1_score(y_val, preds, average='macro')
    return score

In [4]:
logger.info("Starting optimization...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

logger.info("Optimization finished.")
logger.info(f"Best trial score (Macro F1): {study.best_value}")
logger.info(f"Best parameters: {study.best_params}")

2026-04-15 00:41:55,844 - INFO - Starting optimization...
[I 2026-04-15 00:41:55,846] A new study created in memory with name: no-name-1a4291cb-096a-4601-b929-c63502da0525
[I 2026-04-15 00:42:01,307] Trial 0 finished with value: 0.7629249525393531 and parameters: {'n_estimators': 606, 'learning_rate': 0.04977842140866715, 'max_depth': 8, 'num_leaves': 39, 'min_child_samples': 90, 'subsample': 0.8436390136381029, 'colsample_bytree': 0.82724682744872, 'reg_alpha': 4.990823671389696, 'reg_lambda': 4.350086511805015}. Best is trial 0 with value: 0.7629249525393531.
[I 2026-04-15 00:42:02,100] Trial 1 finished with value: 0.7628189575868466 and parameters: {'n_estimators': 660, 'learning_rate': 0.018685414108762385, 'max_depth': 3, 'num_leaves': 43, 'min_child_samples': 87, 'subsample': 0.6679256991288854, 'colsample_bytree': 0.6421263935752145, 'reg_alpha': 1.6643935160729029, 'reg_lambda': 4.240772709811762}. Best is trial 0 with value: 0.7629249525393531.
[I 2026-04-15 00:42:02,522] Tria

In [7]:
def train_final_model(best_params, X_train, y_train, X_validation, y_validation):
    logger.info("Training final model with best parameters...")
    
    final_params = {
        'objective': 'multiclass',
        'num_class': 3,
        'random_state': 42,
        'class_weight': 'balanced',
        **best_params
    }
    
    model = lgb.LGBMClassifier(**final_params)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_validation, y_validation)],
        eval_metric='multi_logloss',
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50)
        ]
    )
    
    return model

# Retrain on full training data, using test data for early stopping validation
final_model = train_final_model(study.best_params, X_train_full, y_train_full, X_test, y_test)

2026-04-15 00:44:16,619 - INFO - Training final model with best parameters...


Training until validation scores don't improve for 50 rounds
[50]	valid_0's multi_logloss: 0.688137
[100]	valid_0's multi_logloss: 0.519154
[150]	valid_0's multi_logloss: 0.438914
[200]	valid_0's multi_logloss: 0.399158
[250]	valid_0's multi_logloss: 0.379233
[300]	valid_0's multi_logloss: 0.368967
[350]	valid_0's multi_logloss: 0.362918
[400]	valid_0's multi_logloss: 0.360266
[450]	valid_0's multi_logloss: 0.359334
[500]	valid_0's multi_logloss: 0.358853
[550]	valid_0's multi_logloss: 0.358282
[600]	valid_0's multi_logloss: 0.357721
[650]	valid_0's multi_logloss: 0.357877
Early stopping, best iteration is:
[610]	valid_0's multi_logloss: 0.357634


In [8]:
def evaluate_model(model, X_test, y_test):
    logger.info("Evaluating final model...")
    preds = model.predict(X_test)
    
    print(f"\nAccuracy: {accuracy_score(y_test, preds)}")
    print("\nClassification Report:\n", classification_report(y_test, preds))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, preds))
    
    logger.info("Saving model...")
    os.makedirs('../../models', exist_ok=True)
    joblib.dump(model, '../../models/lightgbm_churn_model.pkl')
    logger.info("Model saved to ../../models/lightgbm_churn_model.pkl")

evaluate_model(final_model, X_test, y_test)

2026-04-15 00:44:35,479 - INFO - Evaluating final model...
2026-04-15 00:44:35,578 - INFO - Saving model...



Accuracy: 0.8188914910226386

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.66      0.72       427
           1       0.48      0.67      0.56       214
           2       1.00      0.98      0.99       640

    accuracy                           0.82      1281
   macro avg       0.76      0.77      0.75      1281
weighted avg       0.84      0.82      0.83      1281


Confusion Matrix:
 [[280 147   0]
 [ 70 144   0]
 [  6   9 625]]


2026-04-15 00:44:35,815 - INFO - Model saved to ../../models/lightgbm_churn_model.pkl
